# LangChain을 활용한 AI 에이전트 개발 심화
## 2일차 — Middleware, HITL, LangGraph, Multi-Agent, Agentic RAG

핵심 주제: 미들웨어와 가드레일, HITL 승인, LangGraph 상태 관리, 멀티에이전트, Agentic RAG, 관측성과 점검




## 0. 2일차 흐름

1일차에서는 메시지, Context, Tool, Memory, Structured Output을 연결해 기본 에이전트를 만들었습니다.  
2일차에서는 그 에이전트가 더 안전하고, 예측 가능하고, 점검 가능하게 동작하도록 실행 구조를 다룹니다.

오늘 볼 흐름은 다섯 가지입니다.

1. Middleware와 Guardrails로 입력, 모델 호출, Tool 실행을 제어합니다.
2. HITL로 위험한 작업에 사람 승인을 끼워 넣습니다.
3. LangGraph로 상태, 분기, 재시도를 코드로 표현합니다.
4. Multi-Agent로 역할을 나누고, Agentic RAG로 검색 흐름을 고도화합니다.
5. Langfuse와 테스트로 실행 결과를 점검합니다.


In [ ]:
# 2일차 실습에서 사용할 공통 모델을 준비합니다.
import os  # 환경 변수에 저장된 값을 읽습니다.
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv  # .env 파일 값을 현재 세션으로 불러옵니다.
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델을 LangChain에서 사용합니다.

load_dotenv(override=True)  # .env 값을 우선 적용해 현재 노트북 세션으로 올립니다.
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY를 .env에 설정하세요."  # API 키가 없으면 여기서 바로 멈춥니다.

model = ChatOpenAI(model="gpt-4.1-nano")  # 2일차 예제에서도 같은 기본 모델을 사용합니다.
langfuse_handler = None  # Langfuse를 연결하면 여기에 핸들러가 들어갑니다.
lf_config = {}  # callback 등을 함께 넘길 공통 config입니다.
print("모델 준비 완료:", model.model_name)

## 1. 미들웨어

에이전트가 강해질수록 아무 때나 자유롭게 실행되면 위험해집니다.  
예를 들어:

- 이메일 전송
- 데이터베이스 쓰기
- 파일 삭제
- 외부 서비스 호출

같은 작업은 모델이 혼자 판단해서 바로 실행하면 안 될 수 있습니다.

**미들웨어(Middleware)** 는 에이전트 실행 흐름 사이에 끼워 넣는 제어 코드입니다.  
사용자 입력, 모델 호출, Tool 실행, 최종 응답 같은 지점에서 내용을 검사하거나 바꾸거나 실행을 멈출 수 있습니다.

이 절에서는 먼저 Middleware로 실행 흐름을 제어하는 방법을 봅니다.

Middleware가 없으면 모든 규칙을 프롬프트나 Tool 내부 코드에 흩어 넣게 됩니다.  
처음에는 단순해 보여도 민감정보 처리, 승인 대기, 비용 제한, 실행 로그 같은 공통 규칙이 늘어나면 어디서 막고 고쳐야 하는지 찾기 어려워집니다.


<img src="image/middleware_without_vs_with.svg" width="760">


### 1.1 미들웨어는 어디에 끼어드는가?

1일차에서 만든 에이전트는 `모델 호출 -> 필요하면 Tool 실행 -> 최종 답변` 흐름으로 움직였습니다.  
Middleware는 이 흐름의 중간중간에 끼어들어 입력을 손보거나, 실행을 감시하거나, 위험한 동작을 멈추는 장치입니다.




<img src="image/langchain_middleware_flow.png" width="420">

<LangChain 공식 문서의 Middleware flow diagram>




### 1.2 Middleware 훅 정리

| 훅/기능 | 끼어드는 위치 | 주로 쓰는 상황 |
|------|------|------|
| `before_agent` | 에이전트 실행 시작 전 | 요청 초기화, 사용자 권한 확인, 전체 입력 점검 |
| `before_model` | 모델 호출 직전 | 메시지 압축, Context 추가, 위험 입력 차단 |
| `wrap_model_call` | 모델 호출 전체를 감쌈 | 재시도, 폴백 모델, 비용 제한, 모델 선택 |
| `after_model` | 모델 응답 직후, Tool 실행 전 | Tool 호출 검사, HITL 승인 삽입, 응답 필터링 |
| `wrap_tool_call` | Tool 실행 전체를 감쌈 | Tool 인자 검증, 실행 로그, 실패 처리, 위험 Tool 차단 |
| `after_agent` | 에이전트 실행 종료 후 | 최종 응답 기록, 사용량 집계, 감사 로그 저장 |
| `dynamic_prompt` | 모델 호출 전에 시스템 프롬프트를 동적으로 구성 | 사용자 역할, 권한, 현재 상황에 맞는 지시 추가 |

LangChain에서는 위 이름들이 실행 지점을 가리키는 **훅 이름**입니다.  
코드에서도 같은 이름을 메서드나 데코레이터로 볼 수 있습니다.

`before_*`와 `after_*`는 특정 지점 앞뒤에서 실행되고, `wrap_*`은 실제 모델 호출이나 Tool 실행을 감싸서 전후 과정을 함께 제어합니다.  
그래서 Middleware는 프롬프트를 더 길게 쓰는 방식이 아니라, 실행 흐름 자체에 규칙을 끼워 넣는 방식에 가깝습니다.




### 1.3 Middleware가 없을 때 흔히 생기는 문제

에이전트를 처음 만들면 "답을 잘하는지"를 먼저 보게 됩니다.  
하지만 운영에서는 입력, Tool 결과, 승인 같은 중간 지점에서 문제가 생깁니다.

예를 들어 검색 Tool 결과에는 광고 문구, 중복 결과, 오래된 문서, 긴 HTML이 함께 섞일 수 있습니다.  
이 내용이 그대로 모델에 들어가면 토큰 비용이 늘고, 관련 없는 정보 때문에 답변도 흔들립니다.

민감정보가 그대로 전달되거나, 위험한 Tool이 승인 없이 실행되거나, 같은 오류가 계속 반복되는 상황도 마찬가지입니다.

Middleware는 에이전트의 핵심 판단 로직에 모든 예외 처리를 직접 넣는 대신, 입력 처리, Tool 호출 전후,  
모델 응답 전후 같은 실행 흐름의 중간 지점에서 필요한 제어를 수행하는 계층입니다.

<img src="image/Middleware.png" width=600>

Middleware를 사용하면 다음과 같은 처리를 중간에서 수행할 수 있습니다.

- 모델에 전달되기 전 민감정보를 마스킹한다.
- Tool 실행 전에 승인 여부를 확인한다.
- Tool 호출 결과에서 광고, 중복, 노이즈를 제거한다.
- 너무 긴 결과를 요약하거나 필요한 필드만 남긴다.
- 특정 오류가 반복될 때 재시도, 차단, fallback 처리를 수행한다.

즉 Middleware는 에이전트가 모든 입력과 Tool 결과를 그대로 받아들이지 않도록,  
입력과 출력 사이에 두는 제어 계층이라고 볼 수 있습니다.

### 1.4 Guardrails란?

Guardrails는 에이전트가 반드시 지켜야 하는 안전, 품질, 비용 관련 규칙을 의미합니다.

예를 들어 다음과 같은 규칙이 Guardrails에 해당합니다.

- 개인정보를 마스킹한다.
- 허용되지 않은 Tool 호출을 차단한다.
- 위험한 작업은 사람 승인을 받게 한다.
- 응답 형식을 검증한다.
- 과도한 Tool 호출을 제한한다.

LangChain v1에서는 PII 보호, HITL 승인, 호출 제한 같은 Guardrails를 Middleware 형태로 제공하거나 직접 구현할 수 있습니다.

예를 들어 `PIIMiddleware`, `HumanInTheLoopMiddleware`는 각각 개인정보 보호와 승인 절차를 실행 흐름에 적용하는 Middleware입니다.

즉, **Guardrails는 규칙이고 Middleware는 그 규칙을 적용하는 구현 방식**이라고 볼 수 있습니다.

### 1.5 안전성, 비용, 품질은 따로 놀지 않는다

실무에서는 안전성만 챙기고 비용을 무시하거나,  
품질만 높이고 통제를 놓치는 식으로 설계하면 곧 문제가 생깁니다.

좋은 미들웨어 설계는 보통 아래 세 축을 동시에 봅니다.
- **안전성**: 하면 안 되는 일을 막는가?
- **비용**: 불필요한 호출을 줄이는가?
- **품질**: 사용자 경험을 해치지 않는가?

따라서 Guardrails는 단순한 보안 옵션이 아니라 운영 품질 설계의 일부입니다.




### 1.6 PII 가드레일

PII는 개인을 식별할 수 있는 정보입니다. 이메일, 전화번호, 주민등록번호, 주소처럼 그대로 모델에 보내거나 로그에 남기면 위험할 수 있는 값이 여기에 포함됩니다.

PII 가드레일은 모델 호출 전에 이런 값을 찾아 마스킹하거나, 삭제하거나, 아예 요청을 막는 역할을 합니다.  
아래 예제에서는 입력 메시지에 이메일 주소가 들어오면 `PIIMiddleware`가 이를 `[REDACTED_EMAIL]`로 바꿉니다.




#### 1.6.1 자주 쓰는 PII 타입

공식 레퍼런스 기준으로 자주 쓰는 항목은 아래와 같습니다.

| `pii_type` 값 | 의미 | 자주 쓰는 전략 |
|------|------|------|
| `email` | 이메일 주소 | `redact`, `hash` |
| `credit_card` | 신용카드 번호 | `mask`, `block` |
| `ip` | IP 주소 | `redact`, `hash` |
| `mac_address` | MAC 주소 | `redact`, `hash` |
| `url` | URL | `redact`, `block` |




### 1.7 PIIMiddleware는 어느 훅에 속할까?

`PIIMiddleware`는 설정에 따라 끼어드는 위치가 달라집니다.

| 설정 | 해당 훅 | 의미 |
|------|------|------|
| `apply_to_input=True` | `before_model` | 사용자 입력을 모델 호출 전에 검사합니다. |
| `apply_to_output=True` | `after_model` | 모델 응답을 받은 뒤 사용자에게 내보내기 전에 검사합니다. |
| `apply_to_tool_results=True` | `before_model` | Tool 실행 결과가 다음 모델 호출에 들어가기 전에 검사합니다. |

따라서 아래 `email + redact` 예제는 `apply_to_input=True`를 사용하므로 **`before_model` 훅에 해당하는 PII 가드레일**입니다.




#### 1.7.1 PIIMiddleware 주요 옵션

<table>
  <thead>
    <tr>
      <th style='width: 16%;'>옵션</th>
      <th>의미</th>
      <th style='width: 12%;'>기본값</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>pii_type</code></td>
      <td>감지할 PII 종류를 지정합니다. 내장 타입 말고 사용자 정의 이름도 가능합니다.</td>
      <td>필수</td>
    </tr>
    <tr>
      <td><code>strategy</code></td>
      <td>
        감지된 값을 어떻게 처리할지 정합니다.
        <table>
          <thead>
            <tr>
              <th>전략</th>
              <th>처리 방식</th>
              <th>주로 쓰는 상황</th>
            </tr>
          </thead>
          <tbody>
            <tr>
              <td><code>redact</code></td>
              <td><code>[REDACTED_...]</code> 형태로 전체 값을 바꿉니다.</td>
              <td>원문을 남길 필요가 없는 경우</td>
            </tr>
            <tr>
              <td><code>mask</code></td>
              <td>일부 문자만 남기고 나머지를 가립니다.</td>
              <td>카드 번호처럼 끝자리 확인이 필요한 경우</td>
            </tr>
            <tr>
              <td><code>hash</code></td>
              <td>원문 대신 같은 값이면 같은 해시로 바꿉니다.</td>
              <td>사용자를 직접 식별하지 않고 같은 값 여부만 추적할 때</td>
            </tr>
            <tr>
              <td><code>block</code></td>
              <td>요청을 막고 예외를 발생시킵니다.</td>
              <td>민감정보가 있으면 처리 자체를 중단해야 할 때</td>
            </tr>
          </tbody>
        </table>
      </td>
      <td><code>redact</code></td>
    </tr>
    <tr>
      <td><code>detector</code></td>
      <td>정규식이나 함수로 직접 감지 규칙을 만들 때 사용합니다.</td>
      <td><code>None</code></td>
    </tr>
    <tr>
      <td><code>apply_to_input</code></td>
      <td>사용자 입력을 모델 호출 전에 검사합니다.</td>
      <td><code>True</code></td>
    </tr>
    <tr>
      <td><code>apply_to_output</code></td>
      <td>모델 응답을 출력 전에 검사합니다.</td>
      <td><code>False</code></td>
    </tr>
    <tr>
      <td><code>apply_to_tool_results</code></td>
      <td>Tool 실행 결과를 검사합니다.</td>
      <td><code>False</code></td>
    </tr>
  </tbody>
</table>




In [ ]:
# PII 미들웨어는 민감정보를 모델 입력 전에 가려주는 예제입니다.
from langchain.agents import create_agent  # 미들웨어가 붙은 에이전트를 생성합니다.
from langchain.agents.middleware import PIIMiddleware  # 이메일 같은 개인정보를 마스킹합니다.

pii_agent = create_agent(
    model=model,  # 응답을 생성할 기본 모델입니다.
    tools=[],  # 이번 예제는 Tool 없이 입력 보호만 봅니다.
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),  # 이메일을 입력 단계에서 가립니다.
    ],
    system_prompt="민감 정보를 안전하게 처리하는 고객지원 보조 에이전트입니다.",
)

In [ ]:
# 이메일이 모델 입력 전에 가려지는지 확인합니다.
pii_result = pii_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "내 이메일은 minsu.teacher@example.com 이고 강의 자료를 다시 보내줘.",
            }
        ]
    },
    lf_config,
)
print("[마스킹된 입력]")
print(pii_result["messages"][0].content)  # 사용자 입력이 [REDACTED_EMAIL]로 바뀌었는지 확인합니다.
print("\n[에이전트 응답]")
print(pii_result["messages"][-1].content)  # 모델은 마스킹된 입력을 바탕으로 응답합니다.

### 1.8 실습 문제: 신용카드 번호 마스킹하기

앞에서 본 `email + redact` 예제를 바탕으로, 이번에는 `credit_card + mask` 조합을 직접 적용해보세요.

과제 요구사항:
- `PIIMiddleware`의 `pii_type`을 `credit_card`로 설정합니다.
- `strategy`를 `mask`로 설정합니다.
- 입력 문장에는 `5105-1051-0510-5100` 같은 테스트용 카드 번호를 넣습니다.
- 실행 후 카드 번호가 일부만 보이도록 바뀌는지 확인합니다.

주의: 기본 `credit_card` 감지는 숫자 경계를 봅니다.  
`5100이고`처럼 카드번호 바로 뒤에 한국어 조사가 붙으면 감지되지 않을 수 있으므로, 테스트 문장에서는 `5100 이고`처럼 공백을 둡니다.

코드 셀의 `"____"`를 채운 뒤 실행해보세요.




In [ ]:
# `credit_card + mask` 조합으로 입력을 부분 마스킹해보세요.
from langchain.agents import create_agent  # 미들웨어가 붙은 에이전트를 생성합니다.
from langchain.agents.middleware import PIIMiddleware  # 카드 번호 같은 PII를 감지하고 처리합니다.
from langchain_openai import ChatOpenAI  # 실습 셀 단독 실행을 위해 모델을 여기서 만듭니다.

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)  # 마스킹된 입력을 바탕으로 응답할 기본 모델입니다.

In [ ]:
# 빈칸을 채워 카드 번호 마스킹 에이전트를 만듭니다.
mask_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("____", strategy="____", apply_to_input=True),
    ],
    system_prompt="민감 정보를 안전하게 처리하는 고객지원 보조 에이전트입니다.",
)

mask_config = {}  # 추가 실행 설정이 없으면 빈 dict를 넘깁니다.

In [ ]:
# 카드 번호가 부분 마스킹되는지 확인합니다.
mask_result = mask_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "내 법인카드는 5105-1051-0510-5100 이고 결제 내역을 확인해줘.",
            }
        ]
    },
    mask_config,
)

print("[마스킹된 입력]")
print(mask_result["messages"][0].content)
print("\n[에이전트 응답]")
print(mask_result["messages"][-1].content)

<details>
<summary>정답 보기</summary>

```python
from langchain.agents import create_agent  # 미들웨어를 붙인 에이전트를 만듭니다.
from langchain.agents.middleware import PIIMiddleware  # PII를 감지해 전략대로 처리합니다.
from langchain_openai import ChatOpenAI  # 예제에서 사용할 기본 채팅 모델입니다.

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)  # 마스킹된 입력을 읽고 응답할 모델입니다.

mask_agent = create_agent(
    model=model,  # 응답을 생성할 기본 모델입니다.
    tools=[],  # 이번 실습은 Tool 호출 없이 입력 보호만 확인합니다.
    middleware=[
        PIIMiddleware(
            "credit_card",  # 감지 대상은 신용카드 번호입니다.
            strategy="mask",  # 전체를 지우지 않고 일부만 보이게 가립니다.
            apply_to_input=True,  # 모델 호출 전에 사용자 입력을 먼저 검사합니다.
        ),
    ],
    system_prompt="민감 정보를 안전하게 처리하는 고객지원 보조 에이전트입니다.",  # 마스킹된 입력을 기준으로 응답하게 합니다.
)

mask_config = {}  # 추가 실행 설정이 없으면 빈 dict를 넘깁니다.
mask_result = mask_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "내 법인카드는 5105-1051-0510-5100 이고 결제 내역을 확인해줘.",  # 테스트용 카드 번호를 포함한 입력입니다.
            }
        ]
    },
    mask_config,
)

print("[마스킹된 입력]")
print(mask_result["messages"][0].content)  # 카드 번호가 일부만 남도록 가려졌는지 먼저 확인합니다.
print("\n[에이전트 응답]")
print(mask_result["messages"][-1].content)  # 모델은 마스킹된 입력을 바탕으로 응답합니다.
```

확인할 점:
- `mask`는 사람이 읽을 수 있게 일부를 남기고 가립니다.
- 기본 카드번호 감지 규칙은 숫자 앞뒤 경계를 보기 때문에, 카드번호 뒤에는 공백이나 쉼표 같은 구분자를 두는 편이 안전합니다.
- `redact`처럼 전체를 `[REDACTED_...]`로 바꾸는 방식과 다릅니다.

</details>




### 1.9 커스텀 PII 감지 예제

내장 타입만으로 부족할 때는 `detector`에 정규표현식을 넣어 커스텀 PII를 만들 수 있습니다.  
예를 들어 사내 고객번호가 `CUST-1234-5678` 같은 형식이라면, 아래처럼 별도 규칙으로 감지할 수 있습니다.

<details>
<summary>정규표현식이란?</summary>

정규표현식은 문자열에서 특정 패턴을 찾는 규칙입니다.  
예를 들어 `CUST-\d{4}-\d{4}`는 `CUST-` 뒤에 숫자 4개, 하이픈, 숫자 4개가 이어지는 값을 찾습니다.

- `\d`: 숫자 한 자리
- `{4}`: 앞의 패턴이 정확히 4번 반복됨

그래서 `CUST-1234-5678`은 감지되고, `CUST-ABC-5678`은 감지되지 않습니다.

</details>




In [ ]:
# 정규표현식 detector로 커스텀 PII를 감지하는 예제입니다.
from langchain.agents import create_agent  # 미들웨어가 붙은 에이전트를 생성합니다.
from langchain.agents.middleware import PIIMiddleware  # 내장 타입과 커스텀 타입 모두 처리할 수 있습니다.
from langchain_openai import ChatOpenAI  # 실습 셀 단독 실행을 위해 모델을 여기서 만듭니다.

model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)  # 마스킹된 입력을 읽고 응답할 기본 모델입니다.

In [ ]:
# 고객번호 패턴을 감지하는 미들웨어를 붙입니다.
custom_pii_agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware(
            "member_id",  # 커스텀 PII 타입 이름을 직접 붙입니다.
            detector=r"CUST-\d{4}-\d{4}",  # CUST-1234-5678 같은 패턴을 정규표현식으로 찾습니다.
            strategy="redact",  # 감지되면 [REDACTED_MEMBER_ID] 형태로 바꿉니다.
            apply_to_input=True,
        ),
    ],
    system_prompt="민감 정보를 안전하게 처리하는 고객지원 보조 에이전트입니다.",
)

custom_config = {}  # 추가 실행 설정이 없으면 빈 dict를 넘깁니다.

In [ ]:
# 고객번호가 커스텀 PII로 가려지는지 확인합니다.
custom_result = custom_pii_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "고객번호는 CUST-1234-5678이고 배송 상태를 알려줘.",
            }
        ]
    },
    custom_config,
)

print("[마스킹된 입력]")
print(custom_result["messages"][0].content)  # 고객번호가 [REDACTED_MEMBER_ID]로 바뀌는지 확인합니다.
print("\n[에이전트 응답]")
print(custom_result["messages"][-1].content)

### 1.10 Human-in-the-Loop(HITL)이란?

HITL은 에이전트가 어떤 행동을 바로 실행하지 않고, 중간에 사람의 확인을 받도록 멈추는 구조입니다.  
모델이 Tool을 고를 수 있더라도 그 실행 권한까지 항상 자동으로 줘야 하는 것은 아닙니다.

특히 아래 상황에서는 사람이 한 번 판단하도록 두는 편이 안전합니다.
- 발송, 삭제, 결제, 승인 같은 되돌리기 어려운 작업
- 법무, 인사, 재무처럼 책임 소재가 중요한 업무
- 사람이 중간 판단을 내려야 하는 승인형 워크플로




### 1.11 승인 대기형 Tool 예제

아래 예제에서 에이전트는 `send_email` 도구를 사용하려고 시도하지만,  
실제 실행 전에 `HumanInTheLoopMiddleware`가 개입해 흐름을 잠시 중단시킵니다.  
이때 `checkpointer`가 상태를 저장하고, 사용자의 결정이 들어오면 같은 `thread_id`로 이어서 실행됩니다.

이 흐름이 분명하게 보이도록 예시 입력에는 수신자, 제목, 본문을 모두 넣어두었습니다.  
`__interrupt__`가 반환되면 새 요청을 보내기 전에 현재 Tool 호출을 `Command(resume=...)`로 먼저 처리해야 합니다.  
그래야 `reject -> 다시 요청 -> approve` 같은 흐름이 한 대화 안에서 자연스럽게 이어집니다.  
`reject`는 Tool 실행을 막는 결정입니다. 거절 직후에는 같은 Tool 호출을 자동 재시도하지 않고, 사용자가 다시 요청할 때 새 Tool 호출을 만듭니다.  
재요청할 때는 수신자, 제목, 본문을 다시 적어 주는 편이 안정적입니다. 짧은 대명사형 요청은 모델이 단순 확인 응답으로 처리할 수 있습니다.  
`edit`은 Tool 인자를 수정한 뒤 그 수정된 호출을 진행시키는 결정입니다. 별도의 approve 단계가 한 번 더 생기는 것은 아닙니다.

HITL에는 `approve`, `edit`, `reject`, `respond` 같은 decision type이 있을 수 있습니다.  
메일 발송, 파일 삭제, 결제, DB 쓰기처럼 side effect가 있는 Tool에서는 `approve`, `edit`, `reject` 중심으로 제한하는 편이 안전합니다.  
`respond`는 사람이 Tool 결과를 대신 응답하는 흐름에 가깝기 때문에, 이번 메일 발송 예제에서는 제외합니다.




In [ ]:
# HITL 예제에서는 위험한 Tool 실행 전에 사람의 승인을 끼워 넣습니다.
from langchain.tools import tool  # 일반 함수를 Tool로 등록합니다.
from langchain.agents.middleware import HumanInTheLoopMiddleware  # 승인 대기 흐름을 삽입합니다.
from langgraph.checkpoint.memory import InMemorySaver  # 중단 후 재개를 위해 상태를 저장합니다.

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """지정한 수신자에게 이메일을 보냅니다. 실제 발송이므로 신중하게 사용하세요."""
    return f"이메일 발송 완료: to={to}, subject={subject}"  # 여기서는 실제 발송 대신 결과 문자열만 돌려줍니다.

hitl_agent = create_agent(
    model=model,  # Tool 사용 여부를 판단할 모델입니다.
    tools=[send_email],  # 승인 대상 Tool을 등록합니다.
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                }
            }
        )  # side effect가 있는 Tool은 허용할 결정을 명시합니다.
    ],
    checkpointer=InMemorySaver(),  # 중단 전후 상태를 이어가기 위해 필요합니다.
    system_prompt=(
        "이메일 발송 요청이 오면 설명하지 말고 send_email 도구를 바로 사용하세요. "
        "사용자 메시지에 수신자, 제목, 본문이 이미 있으면 추가 질문하지 마세요. "
        "사람이 Tool 실행을 거절했다는 메시지를 받으면 그 즉시 같은 Tool 호출을 반복하지 말고 취소되었다고 짧게 답하세요. "
        "이후 사용자가 메일 발송을 새로 요청하면 새로운 요청으로 보고 send_email 도구를 사용하세요."
    ),
)

In [ ]:
hitl_config = {"configurable": {"thread_id": "email-approval-demo"}}  # 중단 전후를 같은 흐름으로 잇는 ID입니다.
hitl_config.update(lf_config)  # 공통 실행 config를 함께 넘깁니다.

# 첫 호출에서는 Tool이 바로 실행되지 않고 interrupt 정보가 반환됩니다.
first_step = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "support-team@example.com으로, 제목은 'ORD-1001 배송 지연 고객 안내', 본문은 '고객지원팀 여러분, ORD-1001 배송 지연 문의가 접수되었습니다. 고객에게 안내하기 전에 예상 도착일과 보상 쿠폰 여부를 확인해 주세요.' 내용으로 안내 메일을 보내줘.",
            }
        ]
    },
    hitl_config,
)

first_action = first_step["__interrupt__"][0].value["action_requests"][0]

print("[도구 호출 초안]")
print({"name": first_action["name"], "args": first_action["args"]})  # 모델이 만들려던 Tool 호출 인자를 확인합니다.
print("\n[승인 대기 정보]")
print(first_step["__interrupt__"])  # 승인 가능한 Tool과 허용된 결정 종류가 보입니다.


In [ ]:
import pprint
pprint.pprint(first_step['__interrupt__'][0])

In [ ]:
# Command(resume=...)로 승인 또는 거절 결정을 전달하면 중단된 실행이 이어집니다.
from langgraph.types import Command  # interrupt 이후 재개 명령을 표현합니다.

review_decision = {
    "type": "reject",
    "message": "사용자가 이번 메일 발송을 거절했습니다. 현재 중단된 호출은 취소하고, 취소되었다고 짧게 답하세요.",
}  # approve, edit, reject 중 하나를 선택합니다.

resumed = hitl_agent.invoke(
    Command(resume={"decisions": [review_decision]}),
    hitl_config,
)

print("[재개 후 메시지]")
for message in resumed["messages"][-3:]:
    content = message.content or "(내용 없음)"
    print(f"{message.__class__.__name__}: {content}")

In [ ]:
# 거절 후 같은 thread에서 사용자가 다시 요청하면 새 Tool 호출이 만들어지고 다시 승인 대기 상태가 됩니다.
second_step = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "support-team@example.com으로, 제목은 'ORD-1001 배송 지연 고객 안내', 본문은 '고객지원팀 여러분, ORD-1001 배송 지연 문의가 접수되었습니다. 고객에게 안내하기 전에 예상 도착일과 보상 쿠폰 여부를 확인해 주세요.' 내용으로 안내 메일을 다시 보내줘.",
            }
        ]
    },
    hitl_config,
)

second_action = second_step["__interrupt__"][0].value["action_requests"][0]

print("[다시 요청 후 도구 호출 초안]")
print({"name": second_action["name"], "args": second_action["args"]})
print("\n[승인 대기 정보]")
print(second_step["__interrupt__"])


In [ ]:
# 다시 요청으로 생긴 승인 대기 상태를 approve로 재개합니다.
review_decision = {"type": "approve"}  # approve, edit, reject 중 하나를 선택합니다.

approved = hitl_agent.invoke(
    Command(resume={"decisions": [review_decision]}),
    hitl_config,
)

print("[approve 후 메시지]")
for message in approved["messages"][-3:]:
    content = message.content or "(내용 없음)"
    print(f"{message.__class__.__name__}: {content}")

#### edit으로 Tool 인자 수정하기

`approve`는 대기 중인 Tool 호출을 그대로 실행하는 결정입니다.  
`edit`은 사람이 Tool 인자를 고친 뒤, 수정된 호출을 실행하는 결정입니다.

아래 셀에서는 `edit`을 비교해보기 위해 새 승인 대기 상태를 한 번 더 만듭니다.


In [ ]:
# edit도 비교해볼 수 있도록 한 번 더 발송 요청을 만들어 새 승인 대기 상태를 만듭니다.
third_step = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "support-team@example.com으로, 제목은 '[수정] ORD-1001 배송 지연 고객 안내', 본문은 '고객지원팀 여러분, ORD-1001 배송 지연 문의가 접수되었습니다. 고객에게 안내하기 전에 예상 도착일과 보상 쿠폰 여부를 확인해 주세요. 개인정보가 메일 본문에 포함되지 않도록 주의해 주세요.' 내용으로 안내 메일을 다시 보내줘.",
            }
        ]
    },
    hitl_config,
)

third_action = third_step["__interrupt__"][0].value["action_requests"][0]

print("[edit 전 도구 호출 초안]")
print({"name": third_action["name"], "args": third_action["args"]})
print("\n[승인 대기 정보]")
print(third_step["__interrupt__"])


In [ ]:
# edit은 사람이 Tool 인자를 고친 뒤 수정된 호출을 진행시키는 결정입니다.
edit_decision = {
    "type": "edit",
    "edited_action": {
        "name": "send_email",
        "args": {
            "to": "support-team@example.com",
            "subject": "[수정] ORD-1001 배송 지연 고객 안내",
            "body": "고객지원팀 여러분, ORD-1001 배송 지연 문의가 접수되었습니다. 고객에게 안내하기 전에 예상 도착일과 보상 쿠폰 여부를 확인해 주세요. 개인정보가 메일 본문에 포함되지 않도록 주의해 주세요.",
        },
    },
}

edited = hitl_agent.invoke(
    Command(resume={"decisions": [edit_decision]}),
    hitl_config,
)

print("[edit 후 메시지]")
for message in edited["messages"][-3:]:
    content = message.content or "(내용 없음)"
    print(f"{message.__class__.__name__}: {content}")


위 구조의 핵심은 다음과 같습니다.

1. 에이전트는 Tool을 실행하려고 한다.
2. Middleware가 그 호출을 가로챈다.
3. 실행이 중단되고 상태가 저장된다.
4. 사람의 결정 후 같은 `thread_id`로 재개한다.
5. `reject`는 실행을 막고, `edit`은 인자를 고쳐 실행하며, `approve`는 그대로 실행한다.
6. 메일 발송처럼 실제 외부 작업을 실행하는 Tool에서는 `respond`를 열어두지 않는다.

이 패턴은 SQL 실행 승인, 파일 수정 승인, 외부 API 호출 승인에도 그대로 확장할 수 있습니다.




### 1.12 HITL decision type을 어떻게 해석해야 할까?

HITL에서 중요한 것은 단순히 승인 버튼을 누르는 것이 아닙니다.  
실무적으로는 각 결정이 다른 의미를 가집니다.

- **approve**: 시스템이 제안한 행동을 그대로 신뢰한다.
- **edit**: 방향은 맞지만 세부 인자를 사람이 교정하고, 수정된 호출을 진행한다.
- **reject**: 현재 흐름 자체가 부적절하다고 판단한다.
- **respond**: Tool 실행을 건너뛰고 사람이 준 메시지를 Tool 결과처럼 돌려준다.

이메일 발송처럼 side effect가 있는 Tool에는 `approve`, `edit`, `reject`를 명시하고, `respond`는 `ask_user`처럼 사람이 Tool 역할을 대신하는 흐름에만 쓰는 편이 안전합니다.




### 1.13 직접 Middleware 만들기

`PIIMiddleware`, `HumanInTheLoopMiddleware`처럼 자주 쓰는 제어는 내장 Middleware로 먼저 해결합니다.  
내장 옵션으로 맞추기 어려운 처리는 필요한 훅에 직접 붙일 수 있습니다.

아래 예제는 Tool 결과가 다음 모델 호출로 들어가기 전에 광고 문구를 지우고 길이를 줄이는 `wrap_tool_call`입니다.


In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

# wrap_tool_call 미들웨어 함수는 request와 handler를 필수로 받습니다.
# request: 실행하려는 Tool 호출 정보, handler: 실제 Tool을 실행하는 함수입니다.
@wrap_tool_call
def clean_tool_result(request, handler):
    # 원래 Tool을 실행하고 결과를 받습니다.
    result = handler(request)

    # Tool 결과가 문자열이면 다음 모델 호출 전에 짧게 정리합니다.
    if isinstance(result, ToolMessage) and isinstance(result.content, str):
        cleaned = result.content.replace("광고", "")[:1000]
        # ToolMessage로 다시 감싸서 기존 Tool 호출 ID를 유지합니다.
        return ToolMessage(content=cleaned, tool_call_id=result.tool_call_id)

    # ToolMessage가 아니면 그대로 통과시킵니다.
    return result

## 2. LangGraph 기반 워크플로우와 상태 관리

`create_agent()`는 빠르게 에이전트를 만들고 Tool을 붙이는 데 매우 편리합니다.  
하지만 승인, 분기, 재시도, 사람 검토처럼 흐름 자체가 중요해지면, 그 구조를 더 명시적으로 표현할 도구가 필요합니다.

이 절에서는 Agent API를 버리는 것이 아니라, **Agent API로는 어디까지 충분하고 LangGraph가 필요한 지점은 어디인지**를 함께 봅니다.




### 2.1 LangGraph란?

LangGraph는 상태(`State`), 처리 단계(`Node`), 연결(`Edge`)로 에이전트 실행 흐름을 표현하는 도구입니다.  
단순히 한 번 답하고 끝나는 것이 아니라, 분기, 반복, 재시도, 승인 대기처럼 여러 단계를 거치는 흐름을 코드로 직접 제어할 때 사용합니다.

`create_agent()`는 빠르게 에이전트를 만들기 좋은 상위 API입니다.  
하지만 "Tool 호출 뒤에는 반드시 검증 단계를 거친다", "검색 결과가 부족하면 최대 2번만 다시 검색한다", "사람 승인 전에는 발송 노드로 가지 않는다"처럼 실행 순서를 정확히 고정하려면 더 낮은 수준의 설계 도구가 필요합니다.

LangGraph는 그 제어권을 노드와 엣지로 내려서 직접 잡는 방식입니다.  
각 단계가 어떤 상태를 읽고 바꾸는지, 어떤 조건에서 다음 단계로 넘어가는지, 언제 멈추거나 재시도할지를 코드로 명시합니다.

프롬프트에 "실패하면 한 번 더 시도해"라고 적을 수도 있지만, 운영에서는 재시도 횟수, 분기 조건, 종료 지점을 코드로 고정해야 안정적입니다.  
LangGraph는 이런 흐름 규칙을 프롬프트 밖으로 꺼내 고도화된 에이전트 구조로 관리하게 해줍니다.

`create_agent()`로 만든 에이전트도 내부적으로는 LangGraph 기반 런타임 위에서 동작합니다.  
그래서 LangGraph를 알아두면 에이전트가 복잡해질 때 무엇을 더 직접 제어해야 하는지 이해하기 쉬워집니다.

LangGraph는 업무 흐름을 보통 다음 요소로 나누어 생각합니다.
- State: 단계 간 공유되는 데이터
- Node: 처리 단위
- Edge: 흐름 연결
- Conditional Edge: 조건 기반 분기

코드로 옮기면 보통 State는 `class`로 구조를 정의하고, Node는 `state`를 받는 함수로 작성합니다.




공식 문서의 그래프 시각화 예시를 먼저 보면 구조가 더 잘 보입니다.  
박스는 노드, 화살표는 엣지, 갈라지는 화살표는 조건 분기입니다.




<img src="image/langgraph_agentic_rag_graph.png" width="520">

<sub>출처: [LangGraph 공식 문서 - Build a custom RAG agent with LangGraph](https://docs.langchain.com/oss/python/langgraph/agentic-rag)</sub>


### 2.2 아주 간단한 LangGraph 예제

아래 예제는 공지 초안이 너무 짧으면 한 번 더 보완한 뒤 종료하는 아주 단순한 그래프입니다.  
코드는 `상태 정의 -> 노드 정의 -> 엣지 연결 -> 실행 확인` 순서로 읽으면 됩니다.

LangGraph를 처음 볼 때는 문법보다 흐름을 먼저 잡는 편이 좋습니다.

1. 상태(State)를 만듭니다. 그래프가 들고 다닐 작업 기록입니다.
2. 노드(Node)를 만듭니다. 각 노드는 상태를 읽고 한 가지 일을 합니다.
3. 엣지(Edge)로 노드를 연결합니다. 다음에 어떤 노드로 갈지 정합니다.
4. 그래프를 실행합니다. 상태가 노드를 지나면서 조금씩 갱신됩니다.

즉, LangGraph는 "데이터를 들고 다니는 상태"와 "그 상태를 바꾸는 단계들"을 연결해 하나의 실행 흐름으로 만드는 방식입니다.  
`TypedDict`, `add_node`, `add_edge` 같은 코드 문법은 아래 셀에서 필요한 순간에 하나씩 확인합니다.

<img src="image/langgraph_basic_flow.svg" width="760">




#### 2.2.1 상태(State) 정의

`State`는 그래프 안의 여러 노드가 함께 읽고 갱신하는 공용 데이터입니다.  
코드에서는 보통 `TypedDict`로 어떤 키가 오갈지 먼저 적어둡니다. 이렇게 하면 이후 노드와 분기 함수를 읽을 때 훨씬 덜 헷갈립니다.

처음 입력에 모든 값이 들어 있을 필요는 없습니다.  
예를 들어 `draft`, `retry_count`는 처음부터 넣고, `next_step`, `final_text`는 실행 중에 노드가 채웁니다. 코드에서는 이런 값을 `NotRequired`로 표시합니다.

이 예제의 상태값은 다음과 같습니다.
- `draft`: 현재 공지 초안
- `retry_count`: 몇 번 보완했는지
- `next_step`: 다음에 이동할 노드 이름
- `final_text`: 최종 확정된 문장

즉, LangGraph에서는 필요한 값을 메시지 안에만 묻어두지 않고, 상태 객체로 분리해 관리합니다.




In [ ]:
# 먼저 그래프가 공유할 상태 구조를 정의합니다.
from typing_extensions import NotRequired, TypedDict  # 그래프 상태 구조를 타입으로 정의합니다.
from langgraph.graph import StateGraph, START, END  # 그래프 시작점과 종료점을 함께 사용합니다.
from IPython.display import Markdown, display  # Mermaid 그래프를 노트북에 보여줄 때 사용합니다.

class NoticeState(TypedDict):  # 그래프가 단계마다 공유할 상태 구조를 정의합니다.
    draft: str  # 현재 공지 초안 문장입니다.
    retry_count: int  # 초안을 다시 보완한 횟수입니다.
    next_step: NotRequired[str]  # 실행 중에 정해지는 다음 노드 이름입니다.
    final_text: NotRequired[str]  # 최종 단계에서 채워지는 공지 문장입니다.


참고로 같은 상태 구조를 Pydantic `BaseModel`로도 정의할 수 있습니다.  
`TypedDict`는 가볍고 예제 흐름을 읽기 좋고, `BaseModel`은 기본값, 설명, 검증이 필요할 때 유용합니다.

아래 코드는 실행용 셀이 아니라, 같은 `NoticeState`를 Pydantic으로 바꿔 정의한다면 어떤 모습인지 보여주는 참고 예시입니다.  
실제 예제 흐름은 위에서 정의한 `NoticeState(TypedDict)`를 계속 사용합니다.

```python
from pydantic import BaseModel, Field

class NoticeState(BaseModel):
    draft: str = Field(description="현재 공지 초안 문장")
    retry_count: int = Field(default=0, description="초안을 다시 보완한 횟수")
    next_step: str | None = Field(default=None, description="실행 중에 정해지는 다음 노드 이름")
    final_text: str | None = Field(default=None, description="최종 단계에서 채워지는 공지 문장")
```

Pydantic 모델을 State로 쓰면 노드 안에서 `state.draft`처럼 속성으로 값을 읽을 수 있습니다. 노드가 값을 바꿀 때는 TypedDict 예제와 마찬가지로 바뀐 값만 dict로 반환하면 됩니다.




#### 2.2.2 노드(Node) 정의

각 노드는 한 단계의 책임만 맡습니다.  
코드에서는 노드를 함수로 만들고, 함수는 현재 상태를 인자로 받습니다. 노드는 전체 상태를 새로 만들지 않고, 자신이 바꾼 값만 dict로 반환합니다.

노드 함수의 최소 형태는 아래처럼 생각하면 됩니다.

```python
def node_name(state: NoticeState):
    # state에서 필요한 값을 읽는다
    return {"바뀐_상태_키": "새 값"}
```

중요한 규칙은 세 가지입니다.
- 입력 인자는 보통 `state` 하나입니다.
- `state["draft"]`처럼 현재 상태 값을 읽습니다.
- 반환 dict에는 전체 상태가 아니라 이번 노드가 바꾼 값만 넣습니다.

이 예제의 노드는 다음과 같습니다.
- `review_notice`: 초안을 보고 다음 경로를 고릅니다.
- `revise_notice`: 초안을 한 번 더 보완합니다.
- `finalize_notice`: 최종 문장을 확정합니다.
- `route_after_review`: 조건 분기에서 다음 노드 이름을 꺼냅니다.

이렇게 나누면 어느 단계에서 어떤 판단을 했는지 추적하고 수정하기 쉬워집니다.




In [ ]:
# 다음으로 각 노드가 상태를 어떻게 읽고 바꾸는지 정의합니다.
def review_notice(state: NoticeState):  # 초안을 검토해 다음 단계가 무엇인지 결정합니다.
    is_too_short = len(state['draft']) < 20  # 초안이 너무 짧은지 확인합니다.
    can_retry = state.get('retry_count', 0) < 1  # 재실행을 아직 한 번도 하지 않았는지 확인합니다.
    if is_too_short and can_retry:  # 짧고 아직 재실행 가능하면 보완 단계로 보냅니다.
        return {'next_step': 'revise_notice'}  # 다음 노드를 revise_notice로 지정합니다.
    return {'next_step': 'finalize_notice'}  # 아니면 바로 종료 단계로 보냅니다.

def revise_notice(state: NoticeState):  # 초안을 한 번 더 구체적으로 보완합니다.
    return {
        'draft': state['draft'] + ' 예상 도착일과 고객 안내 문구도 함께 적어주세요.',  # 안내 문장을 덧붙입니다.
        'retry_count': state.get('retry_count', 0) + 1,  # 재실행 횟수를 1 올립니다.
    }

def finalize_notice(state: NoticeState):  # 마지막 단계에서 최종 공지 문구를 만듭니다.
    return {'final_text': f"최종 공지: {state['draft']}"}  # 현재 초안을 최종 문장으로 확정합니다.

def route_after_review(state: NoticeState):  # review 단계가 정한 다음 노드 이름을 꺼냅니다.
    return state['next_step']  # add_conditional_edges는 이 문자열을 보고 다음 노드를 고릅니다.


#### 2.2.3 엣지(Edge) 연결

엣지는 노드 사이의 이동 규칙입니다.  
코드에서는 `StateGraph(NoticeState)`로 그래프 빌더를 만들고, `add_node()`로 노드를 등록한 뒤 `add_edge()`로 흐름을 연결합니다. 조건에 따라 다음 노드가 달라질 때는 `add_conditional_edges()`를 씁니다. 이때 조건 함수는 다음에 이동할 노드 이름을 문자열로 반환합니다. 반환 문자열과 실제 노드 이름의 매핑을 함께 넘기면 시각화에서도 분기가 정확히 보입니다.

`START`와 `END`는 실제 업무 단계가 아니라 그래프의 시작점과 종료점을 뜻하는 표시입니다.

이 예제의 흐름은 다음과 같습니다.
- 시작하면 `review_notice`로 들어갑니다.
- 검토 결과가 `revise_notice`면 보완 후 다시 검토합니다.
- 검토 결과가 `finalize_notice`면 종료 단계로 갑니다.

즉, 엣지를 보면 이 그래프가 어떤 순서와 조건으로 움직이는지가 코드로 드러납니다.


In [ ]:
# 이제 노드와 엣지를 연결해 그래프를 만듭니다.
builder = StateGraph(NoticeState)  # NoticeState를 공유하는 그래프 빌더를 만듭니다.
builder.add_node('review_notice', review_notice)  # 검토 노드를 그래프에 등록합니다.
builder.add_node('revise_notice', revise_notice)  # 보완 노드를 그래프에 등록합니다.
builder.add_node('finalize_notice', finalize_notice)  # 종료 직전 노드를 그래프에 등록합니다.

builder.add_edge(START, 'review_notice')  # 시작하면 먼저 review_notice로 들어갑니다.
builder.add_conditional_edges(
    'review_notice',
    route_after_review,
    {
        'revise_notice': 'revise_notice',  # 반환값 -> 실제 이동할 노드
        'finalize_notice': 'finalize_notice',  # 반환값 -> 실제 이동할 노드
    },
)  # route_after_review가 반환한 문자열을 실제 노드 이름에 매핑합니다.
builder.add_edge('revise_notice', 'review_notice')  # 보완 후에는 다시 review 단계로 돌아옵니다.
builder.add_edge('finalize_notice', END)  # 최종 문장을 만들면 그래프를 종료합니다.

notice_graph = builder.compile()  # 실행 가능한 그래프로 컴파일합니다.
notice_graph

#### 2.2.4 실행 결과에서 볼 점

아래 셀을 실행하면 그래프가 실제로 어떤 경로를 타는지 확인할 수 있습니다.
- `review_notice`에서 다음 경로를 고릅니다.
- 초안이 짧으면 `revise_notice`로 갔다가 다시 `review_notice`로 돌아옵니다.
- 마지막에는 `finalize_notice`로 가서 종료합니다.
- 출력된 `notice_result`를 보면 상태가 어떻게 바뀌었는지도 함께 확인할 수 있습니다.




In [ ]:
# 마지막으로 그래프를 실행해 상태가 어떻게 바뀌는지 확인합니다.
notice_result = notice_graph.invoke({'draft': '배송 지연 안내', 'retry_count': 0})  # 짧은 초안을 넣어 실제 흐름을 실행합니다.
print(f"[그래프 실행 결과]\n{notice_result}")  # 상태가 어떻게 바뀌었는지 dict 형태로 확인합니다.
print(f"\n[최종 공지]\n{notice_result['final_text']}")  # 최종 공지 문장만 한 번 더 출력합니다.


### 2.3 LangGraph를 써야 하는 전형적 상황

- 분기 조건이 여러 개 있는 경우
- 실패 시 다른 경로로 우회해야 하는 경우
- 사람 검토 단계를 명시적으로 넣고 싶은 경우
- 반복 루프와 종료 조건을 분리하고 싶은 경우
- "왜 이런 흐름이 나왔는지"를 구조적으로 설명해야 하는 경우




### 2.4 노드를 얼마나 잘게 나눠야 할까?

LangGraph를 설계할 때 자주 나오는 질문이 "노드를 어디까지 쪼개야 하나?"입니다.  
정답은 하나가 아니지만 보통 다음 기준을 씁니다.

- 실패했을 때 별도 재시도가 필요하면 노드를 분리한다.
- 사람이 검토해야 하는 지점이면 노드를 분리한다.
- 외부 시스템 호출이 들어가면 노드를 분리하는 편이 유리하다.
- 항상 함께 움직이는 로직이면 너무 잘게 쪼개지 않아도 된다.

즉, 노드 분해는 코드 스타일 문제가 아니라 **복구성과 설명 가능성의 문제**입니다.




### 2.5 상태 기반 문의 분류 워크플로 예제

아래 예제는 단순하지만 LangGraph 사고방식을 보여줍니다.  
문의 내용을 읽고 긴급도를 분류한 뒤, 일반 응답 초안으로 갈지 사람 검토로 보낼지 분기합니다.

이번에는 그래프를 만드는 과정도 `상태 -> 노드 -> 연결 -> 시각화 -> 실행` 순서로 나눠 봅니다.




#### 2.5.1 문의 상태(State) 정의

이 그래프에서는 문의 원문, 분류된 긴급도, 다음 단계, 최종 응답을 상태로 공유합니다.




In [ ]:
# 문의를 처리하는 동안 공유할 상태 구조를 먼저 정의합니다.
from typing_extensions import NotRequired, TypedDict  # 그래프 상태 구조를 타입으로 정의합니다.
from langgraph.graph import StateGraph, START, END  # 그래프 노드와 시작/종료 지점을 구성합니다.
from IPython.display import Markdown  # Mermaid 그래프를 노트북 안에 표시합니다.

class InquiryState(TypedDict):
    issue: str  # 사용자가 입력한 문의 원문입니다.
    severity: NotRequired[str]  # 분류된 긴급도입니다. normal 또는 high가 들어갑니다.
    next_step: NotRequired[str]  # 다음에 이동할 노드 이름입니다.
    response: NotRequired[str]  # 자동 응답 초안 또는 사람 검토 안내 문장입니다.

#### 2.5.2 문의 처리 노드(Node) 정의

노드는 "분류", "자동 응답", "사람 검토"처럼 한 단계의 책임만 맡습니다.




In [ ]:
# 문의 내용을 읽고 각 단계가 상태를 어떻게 바꾸는지 정의합니다.
def classify_issue(state: InquiryState):
    issue = state["issue"]  # 현재 입력된 문의 문장입니다.
    severity = "high" if any(word in issue for word in ["환불", "중복 결제", "개인정보", "배송 지연"]) else "normal"  # 민감하거나 급한 단어가 있으면 high로 분류합니다.
    next_step = "human_review" if severity == "high" else "draft_reply"  # high면 사람 검토, 아니면 자동 초안으로 보냅니다.
    return {"severity": severity, "next_step": next_step}


def draft_reply(state: InquiryState):
    return {"response": f"자동 초안: 요청을 접수했습니다. 현재 긴급도는 {state['severity']}입니다."}


def human_review(state: InquiryState):
    return {"response": "사람 검토 필요: 환불, 결제, 개인정보, 배송 지연 이슈로 분류되었습니다."}


def route_after_classify(state: InquiryState):
    return state["next_step"]  # classify_issue가 정한 다음 노드 이름을 그대로 반환합니다.

#### 2.5.3 그래프 연결과 시각화

이제 각 노드를 엣지로 연결하고, Mermaid 그래프로 전체 흐름을 확인합니다.




In [ ]:
# 분류 결과에 따라 다른 노드로 가도록 그래프를 연결합니다.
inquiry_builder = StateGraph(InquiryState)  # InquiryState를 공유하는 그래프 빌더를 만듭니다.
inquiry_builder.add_node("classify_issue", classify_issue)  # 문의 긴급도를 분류하는 노드를 등록합니다.
inquiry_builder.add_node("draft_reply", draft_reply)  # 일반 문의 응답 초안을 만드는 노드를 등록합니다.
inquiry_builder.add_node("human_review", human_review)  # 고위험 문의를 사람 검토로 보내는 노드를 등록합니다.

inquiry_builder.add_edge(START, "classify_issue")  # 시작하면 먼저 문의를 분류합니다.
inquiry_builder.add_conditional_edges(
    "classify_issue",
    route_after_classify,
    {
        "draft_reply": "draft_reply",  # 일반 문의면 자동 응답 초안으로 이동합니다.
        "human_review": "human_review",  # 고위험 문의면 사람 검토로 이동합니다.
    },
)  # route_after_classify가 반환한 문자열을 실제 노드 이름에 매핑합니다.
inquiry_builder.add_edge("draft_reply", END)  # 자동 응답을 만들면 그래프를 종료합니다.
inquiry_builder.add_edge("human_review", END)  # 사람 검토 안내를 만들면 그래프를 종료합니다.

inquiry_graph = inquiry_builder.compile()  # 실행 가능한 그래프로 컴파일합니다.


In [ ]:
# 노트북에서는 그래프를 먼저 그려보면 분기 구조를 한눈에 이해하기 쉽습니다.
inquiry_mermaid = inquiry_graph.get_graph().draw_mermaid()
Markdown(f"```mermaid\n{inquiry_mermaid}\n```")

#### 2.5.4 실행 결과 확인

이제 일반 문의와 고위험 문의를 각각 넣어, 서로 다른 경로로 가는지 비교합니다.




In [ ]:
# 일반 요청과 고위험 요청이 서로 다른 분기로 가는지 비교합니다.
normal_case = inquiry_graph.invoke({"issue": "사이즈 교환 방법을 알고 싶습니다."})  # 일반 문의 예시입니다.
high_case = inquiry_graph.invoke({"issue": "중복 결제가 되었고 바로 환불을 요청합니다."})  # 고위험 문의 예시입니다.

print("일반 요청:", normal_case)
print("고위험 요청:", high_case)


이 예제는 단순 규칙 기반이지만,  
실제 서비스에서는 분류 노드만 LLM으로 바꾸고 다른 노드는 결정론적 로직으로 유지하는 식의 **혼합 구조**를 자주 사용합니다.  
그것이 LangGraph가 강한 이유입니다.




### 2.6 Agent API와 Graph API를 비교해보면

`create_agent()`는 모델이 상황에 따라 Tool 사용과 다음 행동을 선택하는 구조를 빠르게 만들 때 적합합니다.  
반면 LangGraph는 승인, 재시도, 분기, 종료 조건처럼 흐름을 코드 수준에서 명시적으로 제어해야 할 때 적합합니다.

- Agent API는 Tool 선택과 행동 결정을 빠르게 실험하기에 좋습니다.
- Graph API는 상태, 분기, 종료 조건을 세밀하게 표현할 수 있습니다.
- 두 API는 경쟁 관계가 아니라, 제어 수준이 다른 도구입니다.

따라서 요구사항이 단순하면 Agent API로 충분하고,  
분기와 상태가 중요해질수록 Graph API가 더 자연스러워집니다.




<img src="image/agent_api_vs_graph_api.svg" width="760">


### 2.7 이걸 create_agent로만 구현하면 어떤 문제가 생길까?

예를 들어 아래 요구사항을 생각해보세요.
- 승인 단계가 있다.
- 실패하면 1회 재시도한다.
- 승인 거절 시 다른 분기로 보낸다.

이 구조를 `create_agent()` 하나로만 처리하려고 하면, 승인 여부, 재시도 횟수, 실패 시 분기 같은 흐름 규칙을 프롬프트와 메시지 상태에 많이 의존하게 됩니다.  
처음에는 돌아가도 운영 단계에서 세 가지 한계가 바로 드러납니다.

- **흐름 제어 어려움**: 어떤 분기에서 왜 다음 행동을 골랐는지 코드 수준에서 고정하기 어렵습니다.
- **상태 관리 어려움**: 승인 여부, 재시도 횟수, 마지막 실패 원인을 메시지에 의존하게 됩니다.
- **디버깅 어려움**: 실패했을 때 프롬프트 문제인지, Tool 문제인지, 경로 문제인지 분리하기 어렵습니다.

정리하면:

- `create_agent()`는 모델이 상황에 따라 Tool 사용과 다음 행동을 선택하는 구조를 빠르게 만들 때 적합합니다.
- LangGraph는 승인, 재시도, 분기, 종료 조건처럼 흐름을 코드 수준에서 명시적으로 제어해야 할 때 적합합니다.

즉, Agent는 **행동 선택**에 강하고,  
LangGraph는 **상태와 흐름 제어**에 강합니다.

<img src="image/create_agent_complex_flow_limit.svg" width="760">


## 3. 멀티에이전트 패턴

한 개의 거대한 에이전트에 모든 역할을 몰아넣으면 컨텍스트가 쉽게 비대해집니다.  
에이전트는 채팅 내용뿐 아니라 어떤 Tool을 호출했는지, Tool 결과가 무엇이었는지, 사람이 어떤 내용을 수정하거나 승인했는지도 함께 참고합니다.  
역할이 많아질수록 이런 기록이 한곳에 계속 쌓여 컨텍스트 창이 생각보다 빨리 차고, 중요한 정보가 뒤로 밀릴 수 있습니다.  
그래서 실무에서는 역할을 나눈 여러 에이전트를 조합합니다.  
각 에이전트가 자기 역할에 필요한 기록과 Tool만 주로 다루게 하면, 컨텍스트가 역할별로 분산되어 더 작고 선명하게 유지됩니다.




아래 그림은 단일 거대 에이전트와 역할별 멀티에이전트 구조의 차이를 보여줍니다.

<img src="image/multiagent_pattern_overview.svg" width="760">


### 3.1 대표 패턴 비교

| 패턴 | 설명 | 적합한 상황 |
|------|------|------|
| Subagents | 메인 에이전트가 전문 에이전트를 Tool처럼 호출 | 전문 영역 분리, 병렬화 |
| Handoffs | 현재 대화 제어권을 다른 에이전트로 넘김 | 순차적 업무 이관 |
| Router | 먼저 분류하고 알맞은 에이전트에 라우팅 | 멀티 도메인 문의 처리 |
| Skills | 단일 에이전트가 필요 시 전문 지식을 로드 | 지식은 분리하되 제어권은 단일화 |




### 3.2 멀티에이전트가 필요한 이유

LangGraph가 한 흐름을 명시적으로 그리는 도구라면,  
멀티에이전트는 그 흐름 안에서 **역할을 나누는 방법**입니다.

핵심은 단계를 나누는 것에서 더 나아가  
책임과 전문성을 어떻게 분리할지 설계하는 것입니다.




### 3.3 Subagents를 Tool처럼 연결하는 예제

Subagent는 특정 역할에 집중하는 작은 에이전트입니다.  
상위 에이전트가 모든 지식과 규칙을 직접 들고 있는 대신, 필요한 순간에 전문 에이전트를 Tool처럼 호출합니다.

아래 예제에서는 정책 에이전트와 일정 에이전트를 각각 별도로 만들고,  
상위 에이전트가 필요에 따라 둘 중 하나를 호출합니다.




### 3.4 에이전트를 Tool처럼 감싸는 구조

정확히는 하위 에이전트가 가진 Tool을 다시 감싸는 것이 아닙니다.  
하위 에이전트를 호출하는 함수 자체를 `@tool`로 만들어, 상위 에이전트가 그 함수를 일반 Tool처럼 고르게 합니다.

- 하위 에이전트 내부 Tool: `search_policy`, `lookup_schedule`
- 상위 에이전트가 보는 Tool: `ask_policy_agent`, `ask_schedule_agent`

질문이 들어오면 supervisor는 `ask_schedule_agent`를 고르고, 그 함수 안에서 `schedule_agent`가 다시 자기 Tool인 `lookup_schedule`을 사용합니다.




<img src="image/multiagent_tool_wrapping.svg" width="760">




#### 3.4.1 하위 에이전트가 쓸 Tool 만들기

먼저 각 전문 에이전트가 직접 사용할 Tool을 만듭니다.  
이 Tool은 supervisor가 직접 쓰는 Tool이 아니라, 하위 에이전트 안에 들어갈 Tool입니다.




In [ ]:
# 먼저 각 전문 영역이 직접 다룰 Tool을 만듭니다.
from langchain.agents import create_agent  # 역할별 에이전트를 생성합니다.
from langchain.tools import tool  # 하위 에이전트 호출을 Tool 형태로 노출합니다.

@tool
def search_policy(query: str) -> str:
    """쇼핑몰 배송, 교환, 환불 규정을 조회합니다."""
    if "환불" in query or "반품" in query:
        return "단순 변심 환불은 상품 수령 후 7일 이내에 신청해야 하며, 반품 배송비는 고객 부담입니다."
    if "교환" in query:
        return "상품 수령 후 7일 이내에 교환을 요청할 수 있습니다."
    return "규정에서 관련 정보를 찾지 못했습니다."

@tool
def lookup_schedule(query: str) -> str:
    """배송 일정과 고객 안내 일정을 조회합니다."""
    if "주말" in query or "배송" in query:
        return "주말 주문은 월요일부터 순차 출고되며, 도서산간 지역은 1~2일 더 걸릴 수 있습니다."
    return "일정 정보가 없습니다."


#### 3.4.2 역할별 전문 에이전트 만들기

각 에이전트는 자기 Tool만 열어 둔 채, 자기 분야 질문에만 집중하게 만듭니다.




In [ ]:
# 규정 전용 에이전트와 일정 전용 에이전트를 따로 만듭니다.
policy_agent = create_agent(
    model=model,  # 규정 질문 전용 모델 실행기입니다.
    tools=[search_policy],  # 규정 Tool만 열어 둡니다.
    system_prompt="쇼핑몰 규정 관련 질문만 답하는 정책 전문가입니다.",
)

schedule_agent = create_agent(
    model=model,  # 일정 질문 전용 모델 실행기입니다.
    tools=[lookup_schedule],  # 일정 Tool만 열어 둡니다.
    system_prompt="배송 일정 관련 질문만 답하는 운영 일정 전문가입니다.",
)

#### 3.4.3 하위 에이전트를 Tool처럼 감싸기

상위 에이전트는 다른 에이전트를 직접 호출하지 않고, Tool처럼 감싼 함수로 위임합니다.




In [ ]:
# 하위 에이전트를 Tool처럼 감싸면 supervisor가 쉽게 호출할 수 있습니다.
@tool
def ask_policy_agent(question: str) -> str:
    """규정 관련 질문을 정책 에이전트에게 위임합니다."""
    result = policy_agent.invoke({"messages": [{"role": "user", "content": question}]}, lf_config)
    return result["messages"][-1].content

@tool
def ask_schedule_agent(question: str) -> str:
    """일정 관련 질문을 일정 에이전트에게 위임합니다."""
    result = schedule_agent.invoke({"messages": [{"role": "user", "content": question}]}, lf_config)
    return result["messages"][-1].content

#### 3.4.4 supervisor 에이전트 실행

마지막으로 상위 에이전트가 질문 성격을 보고, 어떤 하위 에이전트에 넘길지 판단하게 합니다.




In [ ]:
# 상위 에이전트는 질문을 읽고 적절한 하위 에이전트 Tool을 고릅니다.
supervisor = create_agent(
    model=model,  # 상위 에이전트가 어떤 하위 에이전트를 쓸지 판단합니다.
    tools=[ask_policy_agent, ask_schedule_agent],  # 하위 에이전트를 Tool처럼 묶어 둡니다.
    system_prompt="질문의 성격을 보고 적절한 전문 에이전트를 호출하세요.",
)

supervisor_result = supervisor.invoke(
    {"messages": [{"role": "user", "content": "주말 주문 배송 일정은 언제인지 알려줘."}]},
    lf_config,
)
print(supervisor_result["messages"][-1].content)

이 방식의 장점은 메인 에이전트가 모든 세부사항을 직접 기억하지 않아도 된다는 점입니다.  
즉, **컨텍스트를 역할별로 격리**할 수 있어 확장성이 좋아집니다.




### 3.5 언제 멀티에이전트를 쓰지 말아야 할까?

멀티에이전트는 멋져 보이지만, 모든 문제의 해답은 아닙니다.  
오히려 아래 상황에서는 단일 에이전트가 더 낫습니다.

- 작업이 짧고 단순하다.
- 분리된 역할 사이의 경계가 애매하다.
- 병렬화 이점이 거의 없다.
- 디버깅보다 속도가 우선이다.

좋은 설계는 복잡한 구조를 무조건 쓰는 것이 아니라, **최소 구조로 요구사항을 만족하는 것**입니다.




### 3.6 실패하는 멀티에이전트의 전형적인 모습

아래 징후가 보이면 구조가 과해졌을 가능성이 큽니다.
- 동일한 질문을 여러 agent가 반복 처리한다.
- agent 간 응답이 서로 충돌한다.
- delegation이 습관처럼 일어난다.
- hop 수가 늘어 latency가 커진다.
- 어느 단계에서 잘못됐는지 추적하기 어려워진다.

예를 들어 `주말 주문 배송 일정 알려줘` 같은 질문에  
`Router agent -> Schedule agent -> Reviewer agent -> Formatter agent` 순서로 계속 넘기는 구조는 대체로 과합니다.





In [ ]:
# 같은 문제를 단일 Agent + 명확한 Tool 두 개로 단순화할 수 있습니다.
single_ops_agent = create_agent(
    model=model,  # 하나의 에이전트가 두 Tool 중 필요한 것만 고릅니다.
    tools=[search_policy, lookup_schedule],  # 규정 조회와 일정 조회를 같은 에이전트에 둡니다.
    system_prompt=(
        "규정 질문은 search_policy를, 일정 질문은 lookup_schedule을 사용하세요. "
        "둘 다 필요하지 않으면 하나만 호출하세요."
    ),
)

멀티에이전트가 필요한지 판단할 때는 역할 경계가 실제로 분리되는지 먼저 봅니다.  
경계가 약하면 단일 Agent + 좋은 Tool 설계가 더 낫습니다.




## 4. Agentic RAG

RAG(Retrieval-Augmented Generation)는 질문과 관련된 문서를 먼저 검색한 뒤, 그 결과를 바탕으로 답변을 만드는 방식입니다.  
모델이 자기 기억만으로 답하는 대신 외부 문서를 근거로 삼기 때문에, 최신 정보나 사내 문서를 다룰 때 자주 사용합니다.

Naive RAG(기본 RAG)는 보통 `검색 -> 생성` 두 단계로 끝납니다.  
LangChain 문서에서는 이런 고정 흐름을 `2-Step RAG`라고도 부릅니다.  
하지만 실무에서는 검색이 필요한지, 어떤 검색어를 쓸지, 검색 결과가 충분한지 같은 판단이 필요할 때가 많습니다.

Agentic RAG는 RAG 과정 중 일부 의사결정을 LLM이나 Agent가 맡는 구조입니다.  
검색 필요 여부, 검색어 재작성, 검색 결과 평가, 답변 생성 시점 같은 판단 중 일부가 들어가면 Agentic RAG로 볼 수 있습니다.  
반드시 여러 번 검색하거나, 검색 횟수를 모델이 직접 정해야만 하는 것은 아닙니다.

이 단원에서는 같은 문서 검색 문제를 세 단계로 넓혀 봅니다.
- **Naive RAG(기본 RAG)**: 질문으로 벡터 저장소를 한 번 검색하고 답변을 생성합니다.
- **문서 검색 Tool을 붙인 단일 에이전트**: 에이전트가 검색 필요 여부를 판단하고 `retrieve_docs`를 호출합니다.
- **LangGraph로 구현한 Agentic RAG**: 검색, 평가, 재작성, 답변 생성, 종료 조건을 단계로 명시합니다.

여기서 문서 검색 Tool은 웹 검색이 아니라, 앞에서 만든 벡터 저장소를 조회하는 `retrieve_docs`를 뜻합니다.




세 방식의 차이는 검색과 답변 생성을 얼마나 명시적으로 제어하는지입니다.

<img src="image/agentic_rag_overview.svg" width="760">




### 4.1 Agentic RAG를 함께 보는 이유

멀티에이전트 다음에 나오지만, Agentic RAG가 꼭 여러 에이전트를 뜻하는 것은 아닙니다.  
중요한 점은 검색을 고정 함수 호출로만 두지 않고, 검색 필요 여부, 검색어 작성, 결과 평가, 답변 시점 같은 판단을 실행 흐름 안에서 다룬다는 것입니다.

즉, Agentic RAG는 새로운 검색 기법 하나라기보다 **도구 사용, 상태 관리, 흐름 제어가 합쳐진 패턴**으로 보면 이해하기 쉽습니다.




### 4.2 Agentic RAG를 쓰지 말아야 하는 조건

Agentic RAG는 RAG 과정에 판단을 넣어야 할 때 유용하지만, 이미 고정된 검색 흐름으로 충분한 환경에서는 과한 구조가 될 수 있습니다.

다음 조건이면 먼저 Naive RAG(기본 RAG)를 검토하세요.
- 고정된 검색 흐름만으로 품질이 이미 충분한 경우
- 응답 속도가 중요한 서비스
- 비용 제한이 있는 환경

핵심은 "검색을 더 똑똑하게"보다  
"정말 RAG 과정에 추가 의사결정이 필요한가?"를 먼저 보는 것입니다.

추가 판단이 필요 없는 상황에서 Agentic RAG를 쓰면, 성능 개선 없이 비용과 latency만 늘어날 수 있습니다.




### 4.3 LangGraph로 구현한 Agentic RAG 예제

단순 검색형 챗봇은 빠르게 만들 수 있지만,  
검색 품질이 충분하지 않을 때 왜 이런 답이 나왔는지 파악하기 어려운 경우가 많습니다.  
반면 LangGraph로 흐름을 나누면 단계가 명확합니다.

1. 검색한다.
2. 검색 품질을 평가한다.
3. 부족하면 다시 검색하거나 질문을 바꾼다.
4. 충분할 때만 답을 생성한다.

이 구조는 실행 원인을 추적하기 쉽고,  
개선 지점을 찾기에도 유리합니다.




### 4.4 샘플 문서와 검색 저장소 준비

RAG 예제를 보려면 먼저 검색할 문서와 검색 저장소가 필요합니다.  
여기서는 별도 파일이나 데이터베이스 없이 바로 실습할 수 있도록, 코드 안에서 짧은 문서를 만들고 메모리 기반 벡터 저장소에 올립니다.

아래 예제는 로컬 파일 없이도 단일 에이전트형 RAG와 LangGraph로 구현한 Agentic RAG를 비교할 수 있도록,  
가상의 쇼핑몰 고객지원 규정과 관련 없는 내부 문서를 함께 넣어 같은 벡터 저장소를 사용합니다.





In [ ]:
# 쇼핑몰 고객지원 규정과 관련 없는 내부 문서를 함께 검색 대상으로 준비합니다.
from langchain_core.documents import Document  # 검색 대상 문서를 메모리 안에 표현합니다.
from langchain_openai import OpenAIEmbeddings  # 문서를 벡터로 바꿔 유사도 검색에 사용합니다.
from langchain_core.vectorstores import InMemoryVectorStore  # 메모리 기반 벡터 저장소를 만듭니다.

# 예제 문서는 이미 짧은 정책 단위로 나누어 둡니다.
# 관련 없는 문서를 섞어두면 질문과 가까운 문서가 검색되는지 확인하기 쉽습니다.
docs = [
    Document(page_content="무료배송 주문에서 일부 상품을 반품해 최종 구매금액이 50,000원 미만이 되면 최초 배송비 3,000원을 환불액에서 차감한다.", metadata={"source": "partial_return_policy"}),
    Document(page_content="고객이 오배송 또는 상품 불량으로 환불을 요청하면 쇼핑몰이 왕복 배송비를 부담한다. 검수 완료 후 결제 수단으로 전액 환불한다.", metadata={"source": "defect_refund_policy"}),
    Document(page_content="사용한 쿠폰은 주문 취소 후 자동 재발급되지 않을 수 있다. 쿠폰 재발급 가능 여부는 고객센터에서 주문 상태와 쿠폰 종류를 확인한다.", metadata={"source": "coupon_reissue_policy"}),
    Document(page_content="도서산간 지역은 일반 배송보다 1~2일 더 걸릴 수 있다. 배송 지연 문의에는 송장 상태와 예상 도착일을 함께 안내한다.", metadata={"source": "remote_area_shipping_policy"}),
    Document(page_content="사내 회의실 예약은 그룹웨어 캘린더에서 신청하며, 10명 이상 회의는 대회의실을 우선 배정한다.", metadata={"source": "meeting_room_policy"}),
    Document(page_content="임직원 노트북 반출은 보안 포털에서 사전 신청해야 하며, 외부 반출 기간은 최대 7일이다.", metadata={"source": "device_security_policy"}),
    Document(page_content="점심 식대 지원은 평일 근무일 기준으로 제공되며, 야근 식대는 부서장 승인 후 정산한다.", metadata={"source": "meal_expense_policy"}),
    Document(page_content="신규 입사자는 입사 첫 주에 개인정보 보호 교육과 사내 메신저 사용 교육을 완료해야 한다.", metadata={"source": "onboarding_policy"}),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # 문서 의미를 벡터로 바꿀 임베딩 모델입니다.
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)  # 정책 문서를 벡터 저장소에 올립니다.


### 4.5 문서 검색 Tool 준비

이제 벡터 저장소를 바로 호출하는 함수를 하나 만들고, 이를 Tool로 감쌉니다.  
이렇게 하면 에이전트가 필요할 때만 검색을 호출할 수 있습니다.




In [ ]:
# 검색 함수를 Tool로 감싸면 에이전트가 필요할 때만 호출할 수 있습니다.
from langchain.tools import tool  # 검색 함수를 Tool로 노출합니다.

@tool
def retrieve_docs(query: str) -> str:
    """내부 문서에서 쇼핑몰 고객지원 질문과 관련된 내용을 검색합니다."""
    found = vectorstore.similarity_search(query, k=3)  # 관련도가 높은 문서 3개를 찾습니다.
    return "\n\n".join(  # 여러 검색 결과를 빈 줄로 구분해 하나의 문자열로 합칩니다.
        f"[source={doc.metadata.get('source')}] {doc.page_content}"  # 출처와 문서 내용을 함께 보여줍니다.
        for doc in found  # 검색된 문서를 하나씩 문자열로 바꿉니다.
    )

### 4.6 문서 검색 Tool을 붙인 단일 에이전트형 RAG

먼저 가장 단순한 형태를 봅니다.  
에이전트가 질문을 받고, 필요하면 `retrieve_docs`를 호출한 뒤 검색 결과를 근거로 답합니다.

이 예제는 가장 가벼운 Agentic RAG 형태입니다. 에이전트가 검색 필요 여부를 판단하지만,  
질문은 아래 LangGraph Agentic RAG 예제와 같은 쇼핑몰 반품 규정 도메인으로 맞춥니다.  
검색 결과 평가, 재검색, 종료 조건은 아직 코드 흐름으로 분리하지 않았습니다.




In [ ]:
# 먼저 "문서 검색 Tool을 쓰는 단일 에이전트" 형태로 RAG를 경험합니다.
rag_agent = create_agent(
    model=model,  # 답변과 Tool 사용 여부를 판단할 모델입니다.
    tools=[retrieve_docs],  # 문서 검색 Tool 하나만 연결합니다.
    system_prompt="문서 기반 질문에는 retrieve_docs 도구를 사용하고, 검색 결과를 근거로 답변하세요.",
)

rag_result = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "무료배송으로 6만원을 주문했는데 2만원짜리 상품만 반품하면 최초 배송비가 차감되나?"}]},
    lf_config,
)
print(rag_result["messages"][-1].content)


### 4.7 검색-평가-재시도를 LangGraph로 구현하기

여기서는 Agentic RAG의 여러 구현 방식 중 하나로, 같은 RAG 문제를 LangGraph 구조로 다시 구현합니다.  
`rag_graph`는 검색, 관련성 평가, 검색어 재작성, 종료 조건을 각각 노드와 상태로 명시합니다.

그래서 코드가 길어지지만, 어느 단계에서 검색이 부족했는지와 언제 멈췄는지를 추적하기 쉬워집니다.




#### 4.7.1 상태(State)와 평가 스키마 정의

LangGraph로 구현하는 Agentic RAG에서는 질문, 현재 검색어, 검색 문서, 평가 결과, 재시도 횟수처럼  
여러 단계가 함께 써야 하는 값을 상태로 분리해 둡니다.




In [ ]:
# 그래프 안에서 공유할 상태와 관련성 평가 스키마를 먼저 정의합니다.
from typing_extensions import NotRequired, TypedDict  # 노드 사이에 오갈 상태 구조를 정의합니다.
from pydantic import BaseModel, Field  # 관련성 평가 결과를 구조화 출력으로 받습니다.
from langgraph.graph import StateGraph, START, END  # 그래프 노드와 시작/종료 지점을 구성합니다.
from IPython.display import Markdown  # Mermaid 그래프를 노트북 안에 표시합니다.

class RAGState(TypedDict):
    question: str  # 사용자가 입력한 원 질문입니다.
    search_query: NotRequired[str]  # 재작성 후 실제 검색에 사용할 검색어입니다.
    documents: NotRequired[list]  # 벡터 저장소에서 검색된 문서 목록입니다.
    relevant: NotRequired[bool]  # 검색 결과가 질문과 충분히 관련 있는지 평가한 값입니다.
    answer: NotRequired[str]  # 최종 답변 또는 fallback 답변입니다.
    retry_count: int  # 검색어를 다시 써서 재검색한 횟수입니다.
    max_retries: int  # 허용할 최대 재검색 횟수입니다.

class RelevanceCheck(BaseModel):
    relevant: bool = Field(description="검색 문서가 질문과 직접 관련이 있으면 true")  # 관련성 판정 결과를 구조화해 받습니다.

relevance_model = model.with_structured_output(RelevanceCheck)  # 관련성 평가는 구조화 출력으로 단순화합니다.

#### 4.7.2 검색과 재작성 노드

여기서는 Agentic RAG에서 자주 쓰는 의사결정 중 하나인 검색어 재작성과 재검색을 구현합니다.  
질문 그대로 검색해도 되고, 결과가 부족하면 검색어를 바꿔 다시 찾을 수도 있습니다.


In [ ]:
# 질문을 그대로 검색할 수도 있고, 필요하면 검색어를 다시 쓸 수도 있습니다.
def rewrite_query(state: RAGState):
    prompt = f"다음 질문을 검색 친화적으로 다시 써주세요: {state['question']}"
    rewritten = model.invoke(prompt, config=lf_config)  # 질문 자체를 더 검색하기 좋은 표현으로 바꿉니다.
    return {
        "search_query": rewritten.content,
        "retry_count": state.get("retry_count", 0) + 1,  # 재검색 횟수를 1 올립니다.
    }


def retrieve(state: RAGState):
    query = state.get("search_query") or state["question"]  # 재작성된 검색어가 있으면 그것을 우선 사용합니다.
    found = vectorstore.similarity_search(query, k=3)  # 관련도 상위 3개 문서를 찾습니다.
    return {"documents": found}

#### 4.7.3 관련성 평가 노드

검색 후에는 문서가 질문과 충분히 관련 있는지 먼저 평가합니다.  
이 판단이 다음 분기를 결정하는 기준이 됩니다.




In [ ]:
# 검색 결과가 질문과 충분히 관련 있는지 판단합니다.
def grade_documents(state: RAGState):
    joined = "\n\n".join(doc.page_content for doc in state["documents"])  # 평가용으로 문서를 하나로 묶습니다.
    graded = relevance_model.invoke(
        f"질문: {state['question']}\n\n문서:\n{joined}\n\n이 문서들이 질문에 충분히 관련 있나요?"
    )
    return {"relevant": graded.relevant}

#### 4.7.4 답변 생성과 종료 분기

관련성이 충분하면 답을 만들고, 부족하면 재검색으로 돌리거나 안전하게 종료합니다.  
이 예제에서는 "더 검색할지, 여기서 멈출지"라는 의사결정을 그래프 분기로 표현합니다.




In [ ]:
# 평가 결과에 따라 답변 생성, 재검색, 종료를 분기합니다.
def generate_answer(state: RAGState):
    joined = "\n\n".join(doc.page_content for doc in state["documents"])
    answer = model.invoke(
        f"질문: {state['question']}\n\n참고 문서:\n{joined}\n\n위 문서를 기반으로 한국어로 답하세요.",
        config=lf_config,
    )
    return {"answer": answer.content}


def fallback_answer(state: RAGState):
    return {"answer": "검색 결과가 충분하지 않아 추가 확인이 필요합니다. 검색어를 더 구체적으로 바꾸거나 사람이 문서를 확인하세요."}


def route_after_grade(state: RAGState):
    if state["relevant"]:
        return "답변 생성"  # 관련성이 충분하면 바로 답변을 만듭니다.
    if state.get("retry_count", 0) >= state.get("max_retries", 1):
        return "안전 종료"  # 재시도 한도를 넘기면 안전하게 종료합니다.
    return "검색어 재작성"  # 아직 여유가 있으면 검색어를 다시 써봅니다.

#### 4.7.5 그래프 연결과 시각화

이제 노드들을 연결해 실제 Agentic RAG 그래프를 만들고,  
Mermaid 다이어그램으로 한눈에 흐름을 확인합니다.

노드 이름은 다이어그램에 그대로 보이므로, 여기서는 역할이 바로 드러나는 한글 이름을 사용합니다.




In [ ]:
# 검색 -> 평가 -> 재작성/생성 흐름을 그래프로 연결합니다.
rag_graph_builder = StateGraph(RAGState)  # RAGState를 공유하는 그래프 빌더를 만듭니다.
rag_graph_builder.add_node("문서 검색", retrieve)  # 문서를 검색하는 노드를 등록합니다.
rag_graph_builder.add_node("관련성 평가", grade_documents)  # 검색 결과 관련성을 평가하는 노드를 등록합니다.
rag_graph_builder.add_node("검색어 재작성", rewrite_query)  # 검색어를 다시 쓰는 노드를 등록합니다.
rag_graph_builder.add_node("답변 생성", generate_answer)  # 관련 문서로 답변을 생성하는 노드를 등록합니다.
rag_graph_builder.add_node("안전 종료", fallback_answer)  # 재시도 한도 초과 시 안전하게 종료하는 노드를 등록합니다.

rag_graph_builder.add_edge(START, "문서 검색")  # 시작하면 먼저 문서를 검색합니다.
rag_graph_builder.add_edge("문서 검색", "관련성 평가")  # 검색 후 관련성 평가로 이동합니다.
rag_graph_builder.add_conditional_edges(
    "관련성 평가",
    route_after_grade,
    {
        "답변 생성": "답변 생성",  # 관련성이 충분하면 답변 생성으로 이동합니다.
        "검색어 재작성": "검색어 재작성",  # 관련성이 부족하고 재시도 여유가 있으면 검색어를 다시 씁니다.
        "안전 종료": "안전 종료",  # 재시도 한도를 넘기면 안전하게 종료합니다.
    },
)  # route_after_grade가 반환한 문자열을 실제 노드 이름에 매핑합니다.
rag_graph_builder.add_edge("검색어 재작성", "문서 검색")  # 검색어를 바꾼 뒤 다시 검색합니다.
rag_graph_builder.add_edge("답변 생성", END)  # 답변을 생성하면 그래프를 종료합니다.
rag_graph_builder.add_edge("안전 종료", END)  # 안전 종료 답변을 만들면 그래프를 종료합니다.

rag_graph = rag_graph_builder.compile()  # 실행 가능한 그래프로 컴파일합니다.


In [ ]:
# 노트북에서는 그래프 구조를 시각화해두면 흐름을 따라가기가 훨씬 쉽습니다.
rag_graph

#### 4.7.6 실행 결과 확인

그래프를 한 번 실행해보면, 검색어를 그대로 쓸지, 다시 쓸지, 언제 종료하는지를 상태 값으로 확인할 수 있습니다.




In [ ]:
# 실제 질문을 넣어 LangGraph로 만든 Agentic RAG 흐름을 실행합니다.
graph_result = rag_graph.invoke(
    {"question": "무료배송으로 6만원을 주문했는데 2만원짜리 상품만 반품하면 최초 배송비가 차감되나?", "retry_count": 0, "max_retries": 1}  # 예제에서는 재검색을 최대 1번만 허용합니다.
)

print("[최종 검색어]")
print(graph_result.get("search_query") or graph_result["question"])
print("\n[관련성 판정]")
print(graph_result["relevant"])
print("\n[최종 답변]")
print(graph_result["answer"])


이번 예제는 **검색 → 평가 → 재검색/재작성 → 생성** 흐름을 그래프로 표현한 형태입니다.  
실제 서비스에서는 여기에
- 사람이 중간 승인하는 노드
- 쿼리 횟수 제한
- 관측성 태그
- 평가 점수 저장
같은 요소를 더해 운영형 시스템으로 확장합니다.




### 4.8 왜 재시도 횟수 제한이 필요한가?

재검색 루프가 들어간 Agentic RAG는 잘못 설계하면 같은 질문을 계속 바꾸며 무한히 검색하는 구조가 될 수도 있습니다.  
따라서 이런 구조를 쓸 때는 보통 아래 보호장치를 둡니다.

- 최대 재검색 횟수 제한
- 토큰 사용량 상한
- 검색 결과가 일정 기준 이하이면 사람 검토로 전환
- 질문 재작성 실패 시 기본 응답으로 종료

재검색을 넣는다면 단순히 더 많이 검색하게 만드는 것이 아니라 **언제 멈출지도 함께 설계**해야 합니다.




### 4.9 테스트와 관측성이 필요한 이유

여기까지 오면 흐름이 단순 답변을 넘어서 Tool 호출, 분기, 재시도를 포함합니다.  
이런 구조는 결과만 보고는 어느 단계에서 문제가 났는지 알기 어렵습니다.

운영 단계에서는 테스트와 Trace 확인이 필수입니다.




## 5. 테스트와 관측성

에이전트는 결과만 보면 어디서 문제가 생겼는지 알기 어렵습니다.  
그래서 테스트와 실행 기록을 함께 확인합니다.




### 5.1 Langfuse를 왜 보나?

에이전트는 최종 답변만 보면 어디서 문제가 생겼는지 알기 어렵습니다.  
Langfuse를 붙이면 한 번의 요청 안에서 아래 정보를 trace로 확인할 수 있습니다.

- 입력 질문
- Tool 실행 순서
- 모델에 전달된 메시지나 context
- 단계별 지연 시간과 토큰 사용량
- 실행별 비용과 호출 단위 사용량
- 최종 답변까지의 흐름

이 수업처럼 Tool 호출, 분기, 재시도, Agentic RAG까지 다루기 시작하면 trace를 보는 습관이 특히 중요해집니다.




### 5.2 Langfuse란?

Langfuse는 LLM과 Agent 실행을 웹 화면에서 추적하는 관측성 도구입니다.  
처음에는 코드보다 "어떤 화면을 거쳐 설정하는지"를 먼저 잡아두면 덜 헷갈립니다.

보통 순서는 아래처럼 이해하면 됩니다.

1. Organization을 만든다.
2. Project를 만든다.
3. Public Key와 Secret Key를 확인한다.
4. `.env`에 키를 넣는다.
5. `CallbackHandler`를 연결한다.
6. Langfuse 화면에서 trace를 읽는다.

Langfuse는 선택 사항이지만, 흐름이 복잡해질수록 붙여두면 디버깅이 훨씬 쉬워집니다.




#### 5.2.1 Organization 만들기

Langfuse는 프로젝트를 조직 단위로 관리합니다.  
처음 들어가면 먼저 Organization을 만들고, 그 안에서 실습용 프로젝트를 생성하면 됩니다.




Organization 생성 화면

<img src="image/langfuse_organization.png" width="650">




Organization 설정 화면

<img src="image/langfuse_organization2.png" width="650">




#### 5.2.2 Project 만들기

Organization 안에서 새 Project를 만들면, 앞으로 쌓이는 trace가 이 프로젝트 아래에 모입니다.  
실습용과 운영용을 나눠 두면 나중에 trace를 구분해서 보기 편합니다.




Project 생성 화면

<img src="image/langfuse_project.png" width="650">




#### 5.2.3 API Key 확인하기

Project Settings로 들어가면 `Public Key`와 `Secret Key`를 볼 수 있습니다.  
`Public Key`는 프로젝트 식별용, `Secret Key`는 서버 인증용으로 이해하면 됩니다.




Project Settings 화면

<img src="image/langfuse_api_key.png" width="650">




API Key 확인 화면

<img src="image/langfuse_api_key2.png" width="650">




#### 5.2.4 `.env`에 키 넣기

Langfuse Cloud를 쓴다면 `.env`에 아래처럼 넣으면 됩니다.

```bash
LANGFUSE_PUBLIC_KEY=...
LANGFUSE_SECRET_KEY=...
LANGFUSE_BASE_URL=https://cloud.langfuse.com
```

자체 호스팅 환경이라면 `LANGFUSE_BASE_URL`만 사내 Langfuse 주소로 바꾸면 됩니다.




#### 5.2.5 LangChain에 CallbackHandler 연결하기

Langfuse와 LangChain을 연결할 때는 `CallbackHandler`를 사용합니다.  
한 번 만들어 `lf_config`에 넣어두면, 이후 `invoke(..., lf_config)`나 `stream(..., lf_config)`처럼 같은 방식으로 재사용할 수 있습니다.

키를 `.env`에 넣은 뒤 아래 셀을 실행하면 됩니다.




In [ ]:
# Langfuse를 실제로 연결할 때 실행합니다.
from langfuse.langchain import CallbackHandler  # LangChain 실행 정보를 Langfuse trace로 보냅니다.

langfuse_handler = CallbackHandler()  # 이후 invoke나 stream에서 재사용할 핸들러입니다.
lf_config = {"callbacks": [langfuse_handler]}  # 같은 config를 넘기면 trace가 Langfuse에 기록됩니다.

print("Langfuse callback 준비 완료")

#### 5.2.6 Trace 화면에서 먼저 볼 것

실행을 한 번 돌린 뒤 Langfuse에서 trace를 열면, 에이전트가 어떤 순서로 움직였는지 단계별로 볼 수 있습니다.  
처음에는 아래 순서로 읽으면 흐름이 잘 보입니다.

1. 사용자 입력이 무엇이었는지 본다.
2. Tool이나 모델 호출이 몇 번 있었는지 본다.
3. 예상보다 오래 걸린 span이 있는지 본다.
4. 토큰과 비용을 많이 쓴 호출이 어디인지 본다.
5. 기대하지 않은 분기나 재시도가 있었는지 본다.




Trace 상세 화면

<img src="image/langfuse_tracing.png" width="800">




#### 5.2.7 Project 화면에서 Trace 모아보기

한 번의 trace만 보는 것이 아니라, 프로젝트 단위로 여러 실행을 모아 비교할 수도 있습니다.  
같은 질문을 여러 번 돌렸을 때 흐름이 달라졌는지, 어떤 요청에서 병목이 자주 생기는지 볼 때 유용합니다.




Project Trace 목록 화면

<img src="image/langfuse_main.png" width="800">




### 5.3 `GenericFakeChatModel`로 결정론적 테스트하기

관측성이 "실제 실행이 어떻게 흘렀는지"를 보여준다면,  
테스트는 "원하는 흐름이 재현되는지"를 확인하는 장치입니다.

운영 전에 전체 로직을 빠르게 확인하려면 실제 API 대신 가짜 모델을 써서 테스트하는 방법이 유용합니다.  
이렇게 하면 비용 없이, 항상 같은 결과로, CI 친화적인 테스트를 만들 수 있습니다.




In [ ]:
# 가짜 모델을 쓰면 비용 없이 같은 결과를 반복 재현하는 테스트를 만들 수 있습니다.
from langchain_core.language_models import GenericFakeChatModel  # 미리 정한 응답만 돌려주는 테스트용 모델입니다.
from langchain.messages import AIMessage  # 가짜 모델이 반환할 메시지를 만듭니다.
from langchain.agents import create_agent  # 테스트 대상 에이전트를 생성합니다.

fake_model = GenericFakeChatModel(
    messages=iter([
        AIMessage(content="배송 지연 문의는 주문 상태 확인 후 예상 도착일을 안내합니다.")  # 항상 같은 응답을 돌려주도록 고정합니다.
    ])
)

fake_agent = create_agent(
    model=fake_model,  # 실제 API 대신 가짜 모델을 연결합니다.
    tools=[],  # 이번 테스트는 Tool 없이 응답 흐름만 확인합니다.
    system_prompt="배송 지연 문의에 한 문장으로 답하는 에이전트입니다.",
)

fake_result = fake_agent.invoke(
    {"messages": [{"role": "user", "content": "주문 ORD-1001 배송 지연 안내를 짧게 제안해줘."}]}
)
print(fake_result["messages"][-1].content)


### 5.4 운영 전 체크리스트

| 항목 | 확인 포인트 |
|------|-------------|
| Architecture | Agent, Graph, Multi-agent, RAG 구조가 과하지 않은가? |
| Tool 설계 | 이름, docstring, 입력 스키마가 명확한가? |
| HITL | 위험 Tool에 HITL 승인이 적용되어 있는가? |
| Memory | `thread_id`와 상태 범위가 분리되었는가? |
| Graph | LangGraph 분기 또는 상태 관리가 1개 이상 포함되어 있는가? |
| RAG | 검색 결과가 부족할 때 fallback 또는 재검색 제한이 있는가? |
| Observability | Langfuse trace에서 Tool 호출과 중단/재개 흐름을 확인했는가? |
| Test | 최소한의 결정론적 스모크 테스트가 있는가? |




## 6. 구현 연습

아래 guided 실습은 2일차 핵심 요소를 다시 손으로 구현해보는 문제입니다.

목표:
- 위험한 Tool에 승인 대기를 걸어봅니다.
- 같은 `thread_id`로 중단 후 재개를 경험합니다.
- 이 구조가 단순 Agent를 넘어 흐름 제어와 연결된다는 점을 확인합니다.




### 6.1 실습 문제 1

승인 대기형 에이전트를 직접 완성해보세요.

시나리오:
- 사용자가 공지 메일 발송을 요청합니다.
- 에이전트는 바로 발송하지 않고 승인을 기다립니다.
- 승인 후 같은 흐름을 재개합니다.

주의:
- 요청 문장 안에 수신자, 제목, 본문이 모두 들어 있어야 Tool 호출이 바로 일어납니다.
- 메일 발송 Tool에는 `allowed_decisions`를 `approve`, `edit`, `reject`로 제한합니다.

코드 셀의 `None`과 빈칸을 채운 뒤 실행해보세요.




In [ ]:
# Human-in-the-Loop 에이전트를 직접 완성해보세요.
from langchain.agents import create_agent  # Tool을 사용할 에이전트를 만듭니다.
from langchain.tools import tool  # 일반 함수를 Tool로 등록합니다.
from langchain.agents.middleware import HumanInTheLoopMiddleware  # 승인 대기 흐름을 삽입합니다.
from langgraph.checkpoint.memory import InMemorySaver  # 중단 후 재개를 위해 상태를 저장합니다.
from langgraph.types import Command  # resume 명령을 만들 때 사용합니다.

In [ ]:
# 승인 후 실행될 메일 발송 Tool입니다.
@tool
def send_notice_email(to: str, subject: str, body: str) -> str:
    """이 Tool이 언제 사용되고 왜 승인 대상인지 한 문장으로 적어보세요."""
    return None  # 실제 발송 대신 결과 문자열을 반환한다고 가정하고 직접 작성해보세요.

In [ ]:
# 메일 발송 Tool에 Human-in-the-Loop 승인을 겁니다.
approval_agent = create_agent(
    model=model,  # Tool 사용 여부를 판단할 모델입니다.
    tools=[send_notice_email],  # 방금 만든 Tool을 에이전트에 등록합니다.
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_notice_email": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                }
            }
        )  # 메일 발송 Tool에 허용할 결정을 명시해보세요.
    ],
    checkpointer=InMemorySaver(),  # 승인 전후를 같은 흐름으로 이어가기 위해 필요합니다.
    system_prompt=None,  # 인자가 충분하면 추가 질문하지 말고 Tool을 바로 쓰라는 뜻으로 직접 적어보세요.
)


In [ ]:
# 중단 전후를 같은 실행으로 이을 thread_id와 요청 메시지를 준비합니다.
approval_config = {"configurable": {"thread_id": None}}  # 중단 전후를 같은 실행으로 잇는 ID를 직접 적어보세요.
approval_config.update(lf_config)  # 공통 실행 config를 함께 넘깁니다.

request_message = {
    "messages": [
        {
            "role": "user",  # 실제 사용자의 요청 메시지입니다.
            "content": None,  # 수신자, 제목, 본문이 모두 드러나도록 한 문장으로 적어보세요.
        }
    ]
}

In [ ]:
# 위 셀의 None과 빈칸을 채운 뒤 실행하세요.
resume_type = None  # approve, edit, reject 중 하나를 직접 적어보세요.

if not request_message["messages"][0].get("content") or not approval_config["configurable"].get("thread_id") or resume_type is None:
    print("먼저 위 셀의 `None`과 빈칸을 채워주세요.")  # 빈칸 상태로 전체 실행했을 때 헷갈리지 않게 안내합니다.
else:
    paused_result = approval_agent.invoke(request_message, approval_config)
    print(paused_result["__interrupt__"])  # 승인 가능한 Tool과 허용된 결정 종류가 보입니다.

    resumed_result = approval_agent.invoke(
        Command(resume={"decisions": [{"type": resume_type}]}),  # approve, edit, reject 중 하나를 직접 넣어보세요.
        approval_config,
    )

    print("[재개 후 메시지]")
    for message in resumed_result["messages"][-3:]:
        content = message.content or "(내용 없음)"
        print(f"{message.__class__.__name__}: {content}")


<details>
<summary>정답 보기</summary>

```python
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def send_notice_email(to: str, subject: str, body: str) -> str:
    """수신자, 제목, 본문이 정해진 공지 메일을 실제로 발송할 때 사용합니다. 외부 발송이므로 승인 후 실행해야 합니다."""
    return f"공지 메일 발송 완료: to={to}, subject={subject}"

approval_agent = create_agent(
    model=model,
    tools=[send_notice_email],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_notice_email": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                }
            }
        )
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "공지 메일 발송 요청이 오면 설명하지 말고 send_notice_email 도구를 바로 사용하세요. "
        "사용자 메시지에 수신자, 제목, 본문이 이미 있으면 추가 질문하지 마세요."
    ),
)

approval_config = {"configurable": {"thread_id": "hitl-practice-1"}}
approval_config.update(lf_config)
```

확인할 점:
- 첫 번째 호출에서는 바로 메일이 보내지지 않고 중단 정보가 반환됩니다.
- 같은 `thread_id`로 `Command(resume=...)`를 보내야 이어서 실행됩니다.
- 반환 dict의 `__interrupt__`에서 승인 대기 정보를 확인합니다.
- `allowed_decisions`가 `approve`, `edit`, `reject`로 제한되어 있는지 확인합니다.
- `reject`를 넣으면 마지막 AI 응답이 비어 있을 수 있으므로 마지막 몇 개 메시지를 함께 확인합니다.
- 이 구조가 HITL의 핵심입니다.

</details>




## 7. 최종과제

최종과제는 별도 노트북에서 진행합니다.

- 파일: `최종과제.ipynb`
- 주제: 쇼핑몰 고객 문의 처리 에이전트
- 핵심 요소: Tool, Human-in-the-Loop 승인, `thread_id` 기반 후속 질문
- 채점 기준: `최종과제.ipynb` 안의 100점 체크리스트를 사용합니다.
- 점검 방식: 주문 조회 Tool 실행 결과, 승인 대기 상태, 승인 후 재개 결과, 같은 thread_id 기반 후속 질문 결과를 캡처해 제출합니다.
- 선택 점검: Langfuse를 연결한 경우 Trace 화면 캡처를 일부 증거로 활용할 수 있습니다. 단, Langfuse 연결은 필수 제출 조건이 아닙니다.

2일차 본편을 마친 뒤 `최종과제.ipynb`를 열어 안내 순서대로 실행하세요.

## 8. 2일차 정리

| 주제 | 핵심 포인트 |
|------|-------------|
| Middleware | 실행 흐름의 전후를 제어하는 장치 |
| HITL | 위험한 행동을 승인 구조 안에 넣는 방법 |
| create_agent vs LangGraph | Agent는 행동 선택에 강하고, LangGraph는 상태와 흐름 제어에 강함 |
| Multi-Agent | 역할 경계가 분명할 때만 쓰고 과설계를 피함 |
| Agentic RAG | RAG 과정 중 필요한 판단을 LLM/Agent가 맡게 하는 구조 |
| Test / Observability | 운영 전 원인 분석 가능성을 확보 |


